In [ ]:
import sys
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching")
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching\demo-code\2d")
import torch
torch.set_default_device("cuda")
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import h5py
import numpy as np
from rich import print
import math
from tqdm.auto import tqdm

from models import HCDFlowResMLP, HCDFlow
from metrics import sa,l1, l2, pcc

# Load pretrain model

In [ ]:
# model_path = r"E:\Dai hoc\2526I\dacn\flow-matching\run_real_data\checkpoints\tfmemb_nor_8e.pth"
# model = HCDFlow(noise_dim=174, pep_dim=256, time_dim=128, charge_dim=128)
# model.load_state_dict(torch.load(model_path))
model_path = r"E:\Dai hoc\2526I\dacn\flow-matching\run_real_data\checkpoints\tfmemb_adaln6_8e.pth"
model = HCDFlowResMLP(noise_dim=174, pep_dim=256, time_dim=32, charge_dim=8, num_blocks=6)
model.load_state_dict(torch.load(model_path))


model.eval()

## file path

In [ ]:
test_path = r"E:\Dai hoc\2526I\dacn\flow-matching\data\holdout_hcd.hdf5"

# Conditioning test

In [ ]:
with h5py.File(test_path, "r") as f:
    print("Keys:", list(f.keys()))
    total_samples = f["sequence_integer"].shape[0]
    takeaway = 64 # 2 ** 6
    random_idx = np.random.choice(total_samples, size=takeaway, replace=False)
    random_idx.sort()
    fake_seqs = [f["sequence_integer"][0] for _ in range(takeaway)]
    real_seqs = f["sequence_integer"][random_idx]
    fake_charges_oh = [f["precursor_charge_onehot"][0]  for _ in range(takeaway)]
    real_charges_oh = f["precursor_charge_onehot"][random_idx]
    intensities = f["intensities_raw"][random_idx]
    # seqs = f["sequence_integer"][:]
    # charges_oh = f["precursor_charge_onehot"][:]
    # intensities = f["intensities_raw"][:]    

In [ ]:
open("test_index", "w+").write(str(random_idx))

In [ ]:
fake_charges = np.argmax(fake_charges_oh, axis=1) + 1
real_charges = np.argmax(real_charges_oh, axis=1) + 1
del fake_charges_oh, real_charges_oh

In [ ]:
# charges

In [ ]:
intensity_tensors = torch.tensor(intensities, dtype=torch.float32, device="cpu")
fake_seq_tensors = torch.tensor(fake_seqs, dtype=torch.long, device="cpu").expand(takeaway, -1)
real_seq_tensors = torch.tensor(real_seqs, dtype=torch.long, device="cpu")
fake_charge_tensors = torch.tensor(fake_charges, dtype=torch.float32, device="cpu").unsqueeze(1)
real_charge_tensors = torch.tensor(real_charges, dtype=torch.float32, device="cpu").unsqueeze(1)
fake_dataset = TensorDataset(fake_seq_tensors, fake_charge_tensors, intensity_tensors)
real_dataset = TensorDataset(real_seq_tensors, real_charge_tensors, intensity_tensors)

In [ ]:
fake_seq_tensors.shape

In [ ]:
torch.tensor(fake_charges).shape

In [ ]:
batch = 16
total_sample = len(fake_seqs)
total_test = 1

f_loader = DataLoader(
    fake_dataset,
    batch_size=batch,
    shuffle=False,
    pin_memory=True,
    # num_workers=1
)

r_loader = DataLoader(
    real_dataset,
     batch_size=batch,
    shuffle=False,
    pin_memory=True,
    # num_workers=1
)

noises = torch.randn(total_test, intensities.shape[1]).unsqueeze(1).expand(total_test, batch, -1)

In [ ]:
f_sa_losses   = []
f_pcc_losses  = []
r_sa_losses   = []
r_pcc_losses  = []
with torch.no_grad():
    device = torch.get_default_device()
    for seq_b, charge_b, intensity_b in f_loader:
        seq_b = seq_b.to(device, non_blocking=True)
        charge_b = charge_b.to(device, non_blocking=True)
        intensity_b = intensity_b.to(device, non_blocking=True)
        
        # print(seq_b.shape, charge_b.shape, intensity_b.shape)
        for noise in noises:
            # print(noise.shape)
            generated = model.sample(noise, seq_b, charge_b)
            
            f_sa_losses.append(sa(intensity_b, generated))
            f_pcc_losses.append(pcc(intensity_b, generated))
    for seq_b, charge_b, intensity_b in r_loader:
        seq_b = seq_b.to(device, non_blocking=True)
        charge_b = charge_b.to(device, non_blocking=True)
        intensity_b = intensity_b.to(device, non_blocking=True)
        
        # print(seq_b.shape, charge_b.shape, intensity_b.shape)
        for noise in noises:
            # print(noise.shape)
            generated = model.sample(noise, seq_b, charge_b)
            
            r_sa_losses.append(sa(intensity_b, generated))
            r_pcc_losses.append(pcc(intensity_b, generated))

In [ ]:
print(f"Fake: {sum(f_sa_losses[0])/len(f_sa_losses)}")
print(f"Real: {sum(r_sa_losses[0])/len(r_sa_losses)}")

In [ ]:
print(f"Fake: {sum([m for m, ma in f_pcc_losses]) / len(f_pcc_losses)}")
print(f"Real: {sum([m for m, ma in r_pcc_losses]) / len(r_pcc_losses)}")

# Test Full

In [ ]:
with h5py.File(test_path, "r") as f:
    print("Keys:", list(f.keys()))
    seqs = f["sequence_integer"][:]
    charges_oh = f["precursor_charge_onehot"][:]
    intensities = f["intensities_raw"][:]    

In [ ]:
charges = np.argmax(charges_oh, axis=1) + 1
del charges_oh

In [ ]:
num_samples = len(charges)
batch_size = 32
num_batches = math.ceil(num_samples / batch_size)

In [ ]:
test_size = 8
noises = torch.randn(test_size, intensities.shape[1]).unsqueeze(1).expand(test_size, batch_size, -1)

In [ ]:
sa_losses = [[] for _ in range(test_size)] 
pcc_losses = [[] for _ in range(test_size)] 

In [ ]:
test_loop = tqdm(range(num_batches), desc="test")
for b in test_loop:
    start = b * batch_size
    end = min((b + 1) * batch_size, num_samples)

    batch_intensities = torch.tensor(
        intensities[start:end], dtype=torch.float32
    )
    batch_pep_seq = torch.tensor(
        seqs[start:end], dtype=torch.long
    )
    batch_charge = torch.tensor(
        charges[start:end], dtype=torch.float32
    ).unsqueeze(1)

    cur_bs = batch_intensities.shape[0]
    
    for i, noise in enumerate(noises):
        noise = noise[:cur_bs]
        
        # print(noise.shape, batch_pep_seq.shape, batch_charge.shape)
        generated = model.sample(noise, batch_pep_seq, batch_charge, step=8)
        # print(generated.shape, batch_intensities.shape)
        sa_losses[i].append(sa(batch_intensities, generated))
        pcc_losses[i].append(pcc(batch_intensities, generated))
    

In [ ]:
for i, test_res in enumerate(sa_losses):
    print(f"SA loss for test {i+1}: {sum([res[0] for res in test_res])/len(test_res)}")

In [ ]:
for i, test_res in enumerate(pcc_losses):
    print(f"PCC loss for test {i+1}: {sum([res[0] for res in test_res])/len(test_res)}")

In [ ]:
import json
with open("eval_result.json", "w+") as f:
    json.dump({
        "sa": sa_losses,
        "pcc": pcc_losses
    }, f, indent=4)